In [8]:
import json
import urllib.request

# Handling JSON files

In [6]:


def print_persons(filename: str):
    with open(filename) as file:
        persons = json.load(file)
        print(type(persons))
    for person in persons:
        name = person["name"]
        age = person["age"]
        hobbies = ", ".join(person["hobbies"])

        print(f"{name} {age} years ({hobbies})")
print_persons("courses.json")

<class 'list'>
Peter Pythons 27 years (coding, knitting)
Jean Javanese 24 years (coding, rock climbing, reading)


# Course statistics

In [19]:


def retrieve_all():
    address = "https://studies.cs.helsinki.fi/stats-mock/api/courses"

    request = urllib.request.urlopen(address)
    data = request.read()
    print(data)
    courses = json.loads(data)

    result = []

    for course in courses:
        if course["enabled"]:
            exercises_sum = sum(course["exercises"])

            result.append((
                course["fullName"],
                course["name"],
                course["year"],
                exercises_sum
            ))

    return result
retrieve_all()



def retrieve_course(course_name: str):
    address = f"https://studies.cs.helsinki.fi/stats-mock/api/courses/{course_name}/stats"

    request = urllib.request.urlopen(address)
    data = request.read()
    weeks_data = json.loads(data)

    if not weeks_data:
        return {
            "weeks": 0,
            "students": 0,
            "hours": 0,
            "hours_average": 0,
            "exercises": 0,
            "exercises_average": 0
        }

    weeks = len(weeks_data)
    students = max(week["students"] for week in weeks_data.values())
    hours = sum(week["hour_total"] for week in weeks_data.values())
    exercises = sum(week["exercise_total"] for week in weeks_data.values())

    return {
        "weeks": weeks,
        "students": students,
        "hours": hours,
        "hours_average": hours // students if students else 0,
        "exercises": exercises,
        "exercises_average": exercises // students if students else 0
    }
    
print(retrieve_all())
for course in retrieve_all():
    print(retrieve_course(course[1]))

b'[{"week":7,"exercises":[17,13,13,8,6,5,11],"enabled":false,"miniproject":false,"peerReviewOpen":false,"extension":false,"_id":"59f883227655fe0034b4dfe5","year":2017,"term":"syksy","fullName":"Ohjelmistotuotanto","name":"ohtus17","url":"https://github.com/mluukkai/ohjelmistotuotanto2017/wiki/Ohjelmistotuotanto-syksy-2017","__v":7},{"week":8,"exercises":[6,14,19,22,21,21,23,23],"enabled":false,"miniproject":false,"peerReviewOpen":false,"extension":false,"_id":"5a576ac24d91600059c09180","year":1970,"term":"Unknown term","fullName":"Full stack -websovelluskehitys","name":"fs","url":"https://fullstack-hy.github.io","__v":9},{"week":8,"exercises":[6,14,19,22,21,21,23,23],"enabled":false,"miniproject":false,"peerReviewOpen":false,"extension":false,"_id":"5a7f50aa9b73740051c69898","year":2018,"term":"Unknown term","fullName":"Open Full Stack 2018","name":"ofs","url":"http://fullstackopen.github.io","__v":8},{"week":7,"exercises":[0,17,13,13,8,6,6,11],"enabled":false,"miniproject":true,"peerR

# Who cheated

In [24]:

def cheaters():
    start_times = {}

    with open("start_times.csv") as file:
        for line in file:
            name, time = line.strip().split(";")
            hours, minutes = map(int, time.split(":"))
            start_times[name] = hours * 60 + minutes

    cheater_names = set()

    with open("submissions.csv") as file:
        for line in file:
            name, task, points, time = line.strip().split(";")
            hours, minutes = map(int, time.split(":"))
            submission_time = hours * 60 + minutes

            if submission_time - start_times[name] > 180:
                cheater_names.add(name)

    return list(cheater_names)
cheaters()


['jarmo']

# Who cheated, version 2

In [23]:

def final_points():
    start_times = {}

    # Đọc giờ bắt đầu thi
    with open("start_times.csv") as file:
        for line in file:
            name, time = line.strip().split(";")
            hours, minutes = map(int, time.split(":"))
            start_times[name] = hours * 60 + minutes

    # Lưu điểm cao nhất của từng bài
    points = {}

    with open("submissions.csv") as file:
        for line in file:
            name, task, point, time = line.strip().split(";")

            hours, minutes = map(int, time.split(":"))
            submission_time = hours * 60 + minutes

            # Bỏ qua bài nộp muộn hơn 3 tiếng
            if submission_time - start_times[name] > 180:
                continue

            task = int(task)
            point = int(point)

            if name not in points:
                points[name] = {}

            # Chỉ lấy điểm cao nhất của mỗi bài
            if task not in points[name] or point > points[name][task]:
                points[name][task] = point

    # Tính tổng điểm của mỗi sinh viên
    final = {}

    for name in start_times:
        final[name] = sum(points.get(name, {}).values())

    return final
final_points()

{'jarmo': 9, 'timo': 10, 'kalle': 0}

# Spell checker, version 2

In [25]:

from difflib import get_close_matches

# Đọc danh sách từ đúng từ file
with open("wordlist.txt") as file:
    wordlist = [word.strip() for word in file]

# Nhập văn bản
text = input("write text: ")

words = text.split()
corrected_words = []
suggestions = {}

for word in words:
    if word.lower() not in wordlist:
        corrected_words.append("*" + word + "*")
        matches = get_close_matches(word.lower(), wordlist)
        suggestions[word] = matches
    else:
        corrected_words.append(word)

# In câu đã kiểm tra chính tả
print(" ".join(corrected_words))

# In gợi ý
print("suggestions:")
for word, matches in suggestions.items():
    print(f"{word}: {', '.join(matches)}")


*pithon*
suggestions:
pithon: python
